# Tool Selection Accuracy: 3 Ways to Check If Your Agent Picks the Right Tool

Based on: [CCTU: Benchmark for Tool Use under Complex Constraints](https://arxiv.org/abs/2603.15309) (Mar 2026)

## The Problem

Your agent has 10 tools. A user asks "How much does the Marriott cost per night?" The agent should call `get_hotel_pricing`, but it might call `search_hotels` (similar name) or `book_hotel` (related action). With similar tool names, selection errors are common.

The [CCTU paper](https://arxiv.org/abs/2603.15309) found that **no model achieves over 20% task completion** under strict tool constraints. Tool selection is the hardest part.

## What We Compare

We run 10 queries through a live agent with 10 tools, then evaluate with 3 approaches:

| Approach | What It Checks | Cost | Granularity |
|----------|---------------|:----:|------------|
| `ToolCalled` (deterministic) | Was the expected tool called? | Free | Binary per tool |
| `extractors` + manual check | Full tool call list from messages | Free | Full sequence |
| `TrajectoryEvaluator` (LLM) | Was the tool sequence logical? | 1 call | Semantic quality |

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: Run the agent on 10 ground-truth queries

**What this does:** Sends 10 queries to a live agent with 10 available tools, and records which tools the agent actually called for each query.

**What "ground truth" means here:** Each query has one pre-defined correct tool — the tool that a human expert would choose. For example, "How much does the Marriott cost per night?" should use `get_hotel_pricing`, not `search_hotels` or `book_hotel`. This ground truth is established before running the agent and serves as the reference for measuring accuracy.

**Why ground truth matters:** Without ground truth, you cannot measure accuracy — you can only measure consistency. Ground truth turns evaluation from "did the agent do something reasonable?" into "did the agent do the right thing?" The ground truth also documents expected behavior, which is useful for regression testing.

> **What to look for:** For each query, compare the expected tool to the actual tool(s) called. Correct matches get a checkmark. Watch for common errors: similar tool names causing confusion (e.g., `search_hotels` vs `get_hotel_pricing`), or the agent calling multiple tools when only one was needed. The overall accuracy percentage tells you how reliable the agent's tool selection is.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from strands import Agent
from strands.models.openai import OpenAIModel
from strands_evals.extractors import tools_use_extractor
from travel_tools import ALL_TOOLS, GROUND_TRUTH

MODEL = "gpt-4o-mini"

print(f"📋 {len(GROUND_TRUTH)} test queries, {len(ALL_TOOLS)} available tools\n")
print("=" * 70)
print("RUNNING AGENT ON ALL QUERIES")
print("=" * 70)

results = []
for query, expected_tool in GROUND_TRUTH:
    agent = Agent(
        model=OpenAIModel(model_id=MODEL),
        tools=ALL_TOOLS,
        system_prompt="You are a travel assistant. Use the most appropriate tool.",
    )
    response = agent(query)

    # Extract which tools were actually called
    tools_used = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
    actual_tools = [t["name"] for t in tools_used]

    correct = expected_tool in actual_tools
    icon = "✅" if correct else "❌"
    results.append({
        "query": query, "expected": expected_tool,
        "actual": actual_tools, "correct": correct, "response": str(response),
    })
    print(f"  {icon} {query[:50]:<50} expected={expected_tool:<25} got={actual_tools}")

accuracy = sum(1 for r in results if r["correct"]) / len(results)
print(f"\n📊 Agent accuracy: {sum(1 for r in results if r['correct'])}/{len(results)} ({accuracy:.0%})")

---
## Step 2: Evaluate with 3 approaches

We compare three evaluation methods. Each has different strengths:

| Approach | When to Use | Trade-off |
|----------|------------|-----------|
| **`ToolCalled` (deterministic)** | CI/CD gates, regression tests | Fast and free, but binary — only checks if the tool was called, not whether it was the *best* choice |
| **`extractors` + manual check** | Debugging, trajectory analysis | Fast and free, gives full sequence with order and duplicates, but requires you to write the comparison logic |
| **`TrajectoryEvaluator` (LLM)** | Quality reports, nuanced assessment | Can judge semantic appropriateness (e.g., `search_hotels` might be acceptable when `get_hotel_pricing` was expected), but costs 1 LLM call per evaluation |

### Approach A: `ToolCalled` — Deterministic (free, instant)

**What this does:** Checks whether a specific expected tool appears in the list of tools the agent actually called. Returns a binary yes/no.

**Why start here:** This is the cheapest and fastest check. Use it as a CI/CD gate: if the expected tool was not called at all, the test fails immediately without spending tokens on an LLM judge.

> **What to look for:** The accuracy should match what we saw in Step 1. Any discrepancies mean the `ToolCalled` evaluator is interpreting the trajectory data differently from our manual check.

In [ ]:
from strands_evals import Experiment, Case
from strands_evals.evaluators import ToolCalled

print("=" * 70)
print("APPROACH A: ToolCalled (deterministic, free)")
print("=" * 70)

deterministic_correct = 0
for r in results:
    case = Case(name=r["expected"], input=r["query"])
    evaluator = ToolCalled(tool_name=r["expected"])
    exp = Experiment(cases=[case], evaluators=[evaluator])
    # Pass trajectory as list of tool name dicts
    reports = exp.run_evaluations(
        lambda c, traj=r["actual"]: {"output": "", "trajectory": [{"name": t} for t in traj]}
    )
    passed = reports[0].overall_score >= 0.5
    deterministic_correct += passed
    icon = "✅" if passed else "❌"
    print(f"  {icon} ToolCalled('{r['expected']}'): {'found' if passed else 'NOT found'} in {r['actual']}")

print(f"\n📊 ToolCalled accuracy: {deterministic_correct}/{len(results)} ({deterministic_correct/len(results):.0%})")
print(f"   Cost: $0 (deterministic, no LLM calls)")

### Approach B: `TrajectoryEvaluator` — LLM-based (semantic)

**What this does:** An LLM judge evaluates the full tool sequence against a rubric, scoring whether the tools were appropriate for the query.

**When to use this:** Use the LLM-based evaluator when binary matching is too strict. For example, if the expected tool is `get_hotel_pricing` but the agent called `search_hotels` (which also returns prices), the deterministic check fails but the LLM judge may give a partial score because the agent's choice was reasonable, even if not optimal.

> **What to look for:** Compare the LLM scores to the deterministic results. Cases where `ToolCalled` says "fail" but the LLM gives a moderate score (0.5-0.7) indicate queries where the "wrong" tool was still a defensible choice. Cases where both agree on failure are the real errors to investigate.

In [ ]:
from strands_evals.evaluators import TrajectoryEvaluator

traj_eval = TrajectoryEvaluator(
    rubric=(
        "Rate 0-1 whether the agent called the most appropriate tool for the query.\n"
        "1.0: Called the ideal tool directly\n"
        "0.5-0.7: Called a related tool that partially answers the question\n"
        "0.0-0.3: Called an irrelevant tool or missed the correct one"
    ),
    model=MODEL,
)

print("=" * 70)
print("APPROACH B: TrajectoryEvaluator (LLM-based, semantic)")
print("=" * 70)

traj_cases = [
    Case(name=r["expected"], input=r["query"], expected_trajectory=[r["expected"]])
    for r in results
]

def traj_task(case):
    for r in results:
        if r["expected"] == case.name:
            return {"output": r["response"], "trajectory": [{"name": t} for t in r["actual"]]}
    return {"output": "", "trajectory": []}

traj_exp = Experiment(cases=traj_cases, evaluators=[traj_eval])
traj_reports = traj_exp.run_evaluations(traj_task)
traj_reports[0].display()

---
## Comparison Summary

**What this does:** Displays a side-by-side comparison of all three approaches and lists any tool selection errors the agent made.

> **What to look for:** The comparison table summarizes cost, speed, and detection capabilities. Below the table, any errors are listed with the expected vs actual tool — these are the cases worth investigating to improve the agent's system prompt or tool descriptions.

In [ ]:
print("=" * 70)
print("COMPARISON: 3 Approaches to Tool Selection Evaluation")
print("=" * 70)

print(f"""
  ┌────────────────────────┬──────────┬────────────┬─────────────────────┐
  │ Approach               │ Cost     │ Speed      │ What It Catches     │
  ├────────────────────────┼──────────┼────────────┼─────────────────────┤
  │ ToolCalled             │ Free     │ Instant    │ Binary: was it      │
  │ (deterministic)        │          │            │ called yes/no       │
  ├────────────────────────┼──────────┼────────────┼─────────────────────┤
  │ extractors + check     │ Free     │ Instant    │ Full sequence,      │
  │ (deterministic)        │          │            │ order, duplicates   │
  ├────────────────────────┼──────────┼────────────┼─────────────────────┤
  │ TrajectoryEvaluator    │ 1 LLM    │ 2-5s       │ Semantic quality:   │
  │ (LLM judge)            │ call     │            │ was it appropriate? │
  └────────────────────────┴──────────┴────────────┴─────────────────────┘

  💡 Recommendation:
  • Use ToolCalled for CI/CD gates (free, fast, binary)
  • Use extractors for debugging (see full sequence)
  • Use TrajectoryEvaluator for quality reports (semantic, nuanced)

  Errors found (if any):
""")

for r in results:
    if not r["correct"]:
        print(f"  ❌ Query: '{r['query'][:50]}'")
        print(f"     Expected: {r['expected']}, Got: {r['actual']}")
        print()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
fig.set_facecolor('white')

query_labels = [r["query"][:40] + '...' if len(r["query"]) > 40 else r["query"] for r in results]
correctness = [1.0 if r["correct"] else 0.0 for r in results]
bar_colors = ['#4CAF50' if r["correct"] else '#E53935' for r in results]

y_pos = range(len(query_labels))
bars = ax.barh(y_pos, correctness, color=bar_colors, edgecolor='white', height=0.6)

for i, (bar, r) in enumerate(zip(bars, results)):
    label = 'Correct' if r["correct"] else f'Wrong (got {r["actual"]})'
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            label, ha='left', va='center', fontsize=9, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(query_labels, fontsize=9)
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Tool Selection Accuracy per Query\n(Green = correct tool selected, Red = incorrect)', fontweight='bold', fontsize=14)
ax.set_xlim(-0.1, 1.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()